<a href="https://colab.research.google.com/github/EMADUDDINAsdaq/Heart-Disease-Prediction/blob/main/Ada_FFL_%E2%80%94_adaptive_q.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Federated Learning — Method 3: Ada-IFFL (Cong et al. 2023)
Emaduddin Asdaq Syed Mohammed |  CSC8639 MSc Data Science and AI  


---
**This notebook runs q-FedAvg only on the full dataset.**  
Run FedAvg and GIFAIR-FL in separate sessions simultaneously.

**Algorithm 2 — Cong et al. 2023 (Ada-IFFL — Adaptive Individual Fairness FL):**  
Extends q-FedAvg by computing q_k adaptively per client per round.  
Uses Frobenius distance between w_t and w_bar_k to measure model change.  
v_k = 1 - 2*||w_t - w_bar_k|| / (||w_t|| + ||w_bar_k||)  [Eq. 3]  
q_k = beta * |1 - exp(-v_k)|  [Eq. 4]  
Larger model change → larger q_k → more fairness emphasis for that client.  
Note: Ada-IFFL is Algorithm 2; full Ada-FFL (Algorithm 4) adds proximal  
regularisation term mu which requires per-dataset tuning — not implemented here.  
Citation: Cong et al. 2023, CAAI Transactions on Intelligence Technology, DOI: 10.1049/cit2.12232

## Section 1 — Environment Setup

In [ ]:
pip install flwr protobuf

In [ ]:
!pip install "flwr[simulation]" protobuf -q
print("✓ Libraries installed")

✓ Libraries installed


In [ ]:
import flwr as fl
print(f"flwr : {fl.__version__}")
print("✓ Flower working")

flwr : 1.32.0
✓ Flower working


In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score
import flwr as fl
from flwr.common import (NDArrays, parameters_to_ndarrays,
                          ndarrays_to_parameters,
                          FitIns, FitRes, EvaluateIns, EvaluateRes)
from flwr.server.strategy import Strategy
from flwr.server.client_proxy import ClientProxy
from typing import Dict, List, Optional, Tuple
warnings.filterwarnings('ignore')

print(f"flwr     : {fl.__version__}")
print(f"torch    : {torch.__version__}")
print(f"numpy    : {np.__version__}")
print(f"GPU      : {torch.cuda.is_available()}")
print("✓ All libraries ready")

flwr     : 1.32.0
torch    : 2.11.0+cu128
numpy    : 2.0.2
GPU      : True
✓ All libraries ready


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import flwr.simulation
print("Simulation module loaded")

Simulation module loaded


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("=== Session Initialisation ===")
print(f"Device     : {device}")
if torch.cuda.is_available():
    print(f"GPU        : {torch.cuda.get_device_name(0)}")
    print(f"Memory     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print(f"CUDA       : {torch.version.cuda}")
    print("\n✓ GPU ready")
else:
    print("\n⚠ No GPU — Runtime → Change runtime type → A100")

=== Session Initialisation ===
Device     : cuda
GPU        : NVIDIA L4
Memory     : 23.7 GB
CUDA       : 12.8

✓ GPU ready


## Section 2 — Dataset Download and Image Indexing

In [ ]:
import os, time, zipfile

ZIP_PATH     = '/content/drive/MyDrive/dissertation/archive.zip'
EXTRACT      = '/content/nih_unzipped'
DATASET_PATH = EXTRACT

os.makedirs(EXTRACT, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    all_files = z.namelist()
    total     = len(all_files)

print(f"ZIP contains : {total:,} files")
print(f"Extracting to: {EXTRACT}\n")
t0 = time.time()

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    for i, file in enumerate(all_files):
        z.extract(file, EXTRACT)
        pct  = (i + 1) / total * 100
        bar  = '█' * int(pct / 2) + '││' * (50 - int(pct / 2))
        mins = (time.time() - t0) / 60
        print(f"\r[{bar}] {pct:5.1f}% | {i+1:,}/{total:,} | {mins:.1f} min",
              end='', flush=True)

print(f"\n\n✓ Extraction complete in {(time.time()-t0)/60:.1f} minutes")

ZIP contains : 112,128 files
Extracting to: /content/nih_unzipped

[██████████████████████████████████████████████████] 100.0% | 112,128/112,128 | 15.4 min

✓ Extraction complete in 15.4 minutes


In [ ]:
DATASET_PATH = '/content/nih_unzipped'

print("=== NIH ChestX-ray14 — Top Level Contents ===\n")
for item in sorted(os.listdir(DATASET_PATH)):
    p = os.path.join(DATASET_PATH, item)
    if os.path.isdir(p):
        print(f"  [DIR]  {item:<30} {len(os.listdir(p))} items")
    else:
        print(f"  [FILE] {item:<30} {os.path.getsize(p)/1e6:.1f} MB")

=== NIH ChestX-ray14 — Top Level Contents ===

  [FILE] ARXIV_V5_CHESTXRAY.pdf         9.0 MB
  [FILE] BBox_List_2017.csv             0.1 MB
  [FILE] Data_Entry_2017.csv            7.9 MB
  [FILE] FAQ_CHESTXRAY.pdf              0.1 MB
  [FILE] LOG_CHESTXRAY.pdf              0.0 MB
  [FILE] README_CHESTXRAY.pdf           0.8 MB
  [DIR]  images_001                     1 items
  [DIR]  images_002                     1 items
  [DIR]  images_003                     1 items
  [DIR]  images_004                     1 items
  [DIR]  images_005                     1 items
  [DIR]  images_006                     1 items
  [DIR]  images_007                     1 items
  [DIR]  images_008                     1 items
  [DIR]  images_009                     1 items
  [DIR]  images_010                     1 items
  [DIR]  images_011                     1 items
  [DIR]  images_012                     1 items
  [FILE] test_list.txt                  0.4 MB
  [FILE] train_val_list.txt             1.5 MB


In [ ]:
print("Building image path index...")
image_path_dict = {}
for folder in sorted(os.listdir(DATASET_PATH)):
    if folder.startswith('images_'):
        img_dir = os.path.join(DATASET_PATH, folder, 'images')
        if os.path.isdir(img_dir):
            for f in os.listdir(img_dir):
                if f.endswith('.png'):
                    image_path_dict[f] = os.path.join(img_dir, f)
print(f"✓ Indexed {len(image_path_dict):,} images")
sample = '00000001_000.png'
if sample in image_path_dict:
    print(f"✓ Spot check OK : {image_path_dict[sample]}")

Building image path index...
✓ Indexed 112,120 images
✓ Spot check OK : /content/nih_unzipped/images_001/images/00000001_000.png


## Section 3 — Metadata Loading, Exploration and Cleaning

In [ ]:
import pandas as pd

df_raw = pd.read_csv(f"{DATASET_PATH}/Data_Entry_2017.csv")
print(f"Rows    : {len(df_raw):,}")
print(f"Columns : {len(df_raw.columns)}")

Rows    : 112,120
Columns : 12


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
print("=== Missing Values ===\n")
for col in df_raw.columns:
    n = df_raw[col].isnull().sum()
    status = '✓ none' if n == 0 else f'⚠  {n:,} missing'
    print(f"  {col:<35} {status}")

=== Missing Values ===

  Image Index                         ✓ none
  Finding Labels                      ✓ none
  Follow-up #                         ✓ none
  Patient ID                          ✓ none
  Patient Age                         ✓ none
  Patient Gender                      ✓ none
  View Position                       ✓ none
  OriginalImage[Width                 ✓ none
  Height]                             ✓ none
  OriginalImagePixelSpacing[x         ✓ none
  y]                                  ✓ none
  Unnamed: 11                         ⚠  112,120 missing


In [ ]:
import numpy as np

df = df_raw.copy()
df = df.rename(columns={'Patient Gender': 'Patient Sex'})
print("✓ Step 1 — Patient Gender → Patient Sex")

null_cols = [c for c in df.columns if df[c].isnull().all()]
df        = df.drop(columns=null_cols)
print(f"✓ Step 2 — Removed null columns: {null_cols}")

before = len(df)
df     = df[df['Patient Age'] <= 100]
print(f"✓ Step 3 — Removed {before - len(df)} rows (age > 100)")

df['label'] = (df['Finding Labels'] != 'No Finding').astype(int)
print(f"✓ Step 4 — Binary label | Pathology rate: {df['label'].mean()*100:.1f}%")

df['Age Group'] = pd.cut(
    df['Patient Age'],
    bins   = [0, 20, 40, 60, 80, 101],
    labels = ['0-20', '20-40', '40-60', '60-80', '80+']
)
print("✓ Step 5 — Age groups added")

df['image_path'] = df['Image Index'].map(image_path_dict)
print(f"✓ Step 6 — Image paths linked: {df['image_path'].notna().sum():,} matched")
print(f"\nFinal dataset : {len(df):,} rows")

✓ Step 1 — Patient Gender → Patient Sex
✓ Step 2 — Removed null columns: ['Unnamed: 11']
✓ Step 3 — Removed 16 rows (age > 100)
✓ Step 4 — Binary label | Pathology rate: 46.2%
✓ Step 5 — Age groups added
✓ Step 6 — Image paths linked: 112,104 matched

Final dataset : 112,104 rows


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Section 4 — GPU Optimisation

In [ ]:
import flwr as fl
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score
from typing import Dict, List, Optional, Tuple
from flwr.common import (NDArrays, Scalar, Parameters,
                          parameters_to_ndarrays, ndarrays_to_parameters,
                          FitIns, FitRes, EvaluateIns, EvaluateRes)
from flwr.server.strategy import Strategy
from flwr.server.client_proxy import ClientProxy
import numpy as np, warnings, json, os, time, copy
warnings.filterwarnings('ignore')

print(f"✓ Flower : {fl.__version__}")
print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ GPU    : {torch.cuda.is_available()}")

✓ Flower : 1.32.0
✓ PyTorch: 2.11.0+cu128
✓ GPU    : True


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"Memory : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print("\n✓ GPU confirmed")

Device : cuda
GPU    : NVIDIA L4
Memory : 23.7 GB

✓ GPU confirmed


In [ ]:
torch.backends.cudnn.benchmark     = True
torch.backends.cudnn.deterministic = False

BATCH_SIZE  = 512
NUM_WORKERS = 4
PREFETCH    = 2

print(f"✓ BATCH_SIZE  : {BATCH_SIZE}")
print(f"✓ NUM_WORKERS : {NUM_WORKERS}")
print(f"✓ PREFETCH    : {PREFETCH}")
print(f"✓ cuDNN bench : enabled")

✓ BATCH_SIZE  : 512
✓ NUM_WORKERS : 4
✓ PREFETCH    : 2
✓ cuDNN bench : enabled


## Section 5 — Dirichlet Partition → 5 Hospital Clients

In [ ]:
np.random.seed(42)

NUM_CLIENTS    = 5
ALPHA          = 0.5
HOSPITAL_NAMES = ['Hospital_A', 'Hospital_B', 'Hospital_C',
                  'Hospital_D', 'Hospital_E']

print(f"Clients : {NUM_CLIENTS}")
print(f"Alpha   : {ALPHA}  — Dir(α=0.5) [Morafah et al. 2022]")
print(f"Seed    : 42")

Clients : 5
Alpha   : 0.5  — Dir(α=0.5) [Morafah et al. 2022]
Seed    : 42


In [ ]:
idx_0 = np.where(df['label'].values == 0)[0]
idx_1 = np.where(df['label'].values == 1)[0]
np.random.shuffle(idx_0)
np.random.shuffle(idx_1)

props_0 = np.random.dirichlet(np.repeat(ALPHA, NUM_CLIENTS))
props_1 = np.random.dirichlet(np.repeat(ALPHA, NUM_CLIENTS))

splits_0 = (props_0 * len(idx_0)).astype(int)
splits_1 = (props_1 * len(idx_1)).astype(int)
splits_0[-1] = len(idx_0) - splits_0[:-1].sum()
splits_1[-1] = len(idx_1) - splits_1[:-1].sum()

clients = {}
ptr0, ptr1 = 0, 0
for i, name in enumerate(HOSPITAL_NAMES):
    idx = np.concatenate([
        idx_0[ptr0 : ptr0 + splits_0[i]],
        idx_1[ptr1 : ptr1 + splits_1[i]]
    ])
    clients[name] = df.iloc[idx].copy().reset_index(drop=True)
    ptr0 += splits_0[i]
    ptr1 += splits_1[i]

print("=" * 80)
print(f"HOSPITAL CLIENT STATISTICS — Dirichlet Dir(α={ALPHA}) [Morafah et al. 2022]")
print("=" * 80)
print(f"\n{'Client':<14} {'Images':>8} {'Pathology%':>12} {'No Finding%':>13} {'Male%':>7}")
print("-" * 57)
for name, data in clients.items():
    path = data['label'].mean() * 100
    male = (data['Patient Sex'] == 'M').mean() * 100
    print(f"{name:<14} {len(data):>8,} {path:>11.1f}% {100-path:>12.1f}% {male:>6.1f}%")
print("\n✓ Non-IID partition confirmed")

HOSPITAL CLIENT STATISTICS — Dirichlet Dir(α=0.5) [Morafah et al. 2022]

Client           Images   Pathology%   No Finding%   Male%
---------------------------------------------------------
Hospital_A       61,505        59.9%         40.1%   56.4%
Hospital_B       10,655        99.7%          0.3%   56.8%
Hospital_C       34,936         8.2%         91.8%   56.6%
Hospital_D        1,495        69.1%         30.9%   58.0%
Hospital_E        3,513        10.0%         90.0%   55.6%

✓ Non-IID partition confirmed


## Section 6 — PyTorch Setup: Transforms, Dataset, Model

In [ ]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

print(f"✓ Image size    : {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"✓ Normalisation : ImageNet mean/std [Zech et al. 2018]")

✓ Image size    : 224×224
✓ Normalisation : ImageNet mean/std [Zech et al. 2018]


In [ ]:
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(row['label'], dtype=torch.float32)
        return image, label

print("✓ ChestXrayDataset defined")

✓ ChestXrayDataset defined


In [ ]:
ROUNDS = 10

def build_model():
    model    = models.resnet18(weights='IMAGENET1K_V1')
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model

test_model = build_model()
params     = sum(p.numel() for p in test_model.parameters())
print(f"✓ ResNet-18 — ImageNet pretrained [Zech et al. 2018]")
print(f"✓ Final layer : 512 → 1 (binary classification)")
print(f"✓ Parameters  : {params:,}")
print(f"✓ Rounds      : {ROUNDS}")
del test_model

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 238MB/s]


✓ ResNet-18 — ImageNet pretrained [Zech et al. 2018]
✓ Final layer : 512 → 1 (binary classification)
✓ Parameters  : 11,177,025
✓ Rounds      : 10


In [ ]:
SAMPLE_FRACTION = 1.00
TRAIN_SPLIT     = 0.85
VAL_SPLIT       = 0.10
# remaining 0.05 = test

train_clients = {}
val_clients   = {}
test_clients  = {}

for name, data in clients.items():
    data_sample = data.sample(
        frac=SAMPLE_FRACTION, random_state=42
    ).reset_index(drop=True)

    n       = len(data_sample)
    n_train = int(n * TRAIN_SPLIT)
    n_val   = int(n * VAL_SPLIT)

    train_clients[name] = data_sample.iloc[:n_train].reset_index(drop=True)
    val_clients[name]   = data_sample.iloc[n_train:n_train+n_val].reset_index(drop=True)
    test_clients[name]  = data_sample.iloc[n_train+n_val:].reset_index(drop=True)

print(f"=== 10% Dataset | 85/10/5 Split ===\n")
print(f"{'Client':<14} {'Train':>8} {'Val':>6} {'Test':>6} {'Pathology%':>12}")
print("-" * 50)
for name in HOSPITAL_NAMES:
    print(f"{name:<14} {len(train_clients[name]):>8,} "
          f"{len(val_clients[name]):>6,} "
          f"{len(test_clients[name]):>6,} "
          f"{train_clients[name]['label'].mean()*100:>11.1f}%")
print(f"\n✓ train/val/test splits ready — 85/10/5")

=== 10% Dataset | 85/10/5 Split ===

Client            Train    Val   Test   Pathology%
--------------------------------------------------
Hospital_A       52,279  6,150  3,076        60.0%
Hospital_B        9,056  1,065    534        99.7%
Hospital_C       29,695  3,493  1,748         8.3%
Hospital_D        1,270    149     76        68.6%
Hospital_E        2,986    351    176         9.7%

✓ train/val/test splits ready — 85/10/5


## Section 7 — Evaluation Function

In [ ]:
def evaluate_client(model, dataframe, device):
    model = model.to(device)
    model.eval()

    loader = DataLoader(
        ChestXrayDataset(dataframe, transform=val_transform),
        batch_size         = BATCH_SIZE,
        shuffle            = False,
        num_workers        = NUM_WORKERS,
        pin_memory         = True,
        persistent_workers = True
    )

    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            probs = torch.sigmoid(
                model(images.to(device, non_blocking=True))
            ).cpu().numpy()
            all_probs.extend(probs.flatten())
            all_labels.extend(labels.numpy())

    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    preds  = (probs > 0.5).astype(int)

    def auc_fnr_for_mask(y_true, y_prob, y_pred):
        if len(y_true) < 10 or y_true.sum() == 0:
            return float('nan'), float('nan')
        try:
            auc = float(roc_auc_score(y_true, y_prob))
        except Exception:
            auc = float('nan')
        fn  = int(((y_pred == 0) & (y_true == 1)).sum())
        tp  = int(((y_pred == 1) & (y_true == 1)).sum())
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
        return round(auc, 4), round(fnr, 4)

    auc, fnr = auc_fnr_for_mask(labels, probs, preds)
    acc      = round(float((preds == labels).mean() * 100), 2)
    metrics  = {'auc': auc, 'fnr': fnr, 'accuracy': acc}

    for sex in ['M', 'F']:
        mask = dataframe['Patient Sex'].values == sex
        if mask.sum() > 10:
            a, f = auc_fnr_for_mask(labels[mask], probs[mask], preds[mask])
            metrics[f'auc_{sex}'] = a
            metrics[f'fnr_{sex}'] = f

    for grp in ['0-20', '20-40', '40-60', '60-80', '80+']:
        mask = dataframe['Age Group'].values == grp
        if mask.sum() > 10:
            a, f = auc_fnr_for_mask(labels[mask], probs[mask], preds[mask])
            metrics[f'auc_{grp}'] = a
            metrics[f'fnr_{grp}'] = f

    return metrics

print("✓ evaluate_client() defined")
print("  Metrics : AUC + FNR (overall, per sex, per age group)")
print("  Citation: Seyyed-Kalantari et al. 2021 | Ahluwalia et al. 2023")

✓ evaluate_client() defined
  Metrics : AUC + FNR (overall, per sex, per age group)
  Citation: Seyyed-Kalantari et al. 2021 | Ahluwalia et al. 2023


## Section 8 — Flower Base Client

In [ ]:
class HospitalClient(fl.client.NumPyClient):

    def __init__(self, name: str, dataframe, val_dataframe, device):
        self.name          = name
        self.dataframe     = dataframe
        self.val_dataframe = val_dataframe
        self.device        = device
        self.model         = build_model().to(device)

    def get_parameters(self, config) -> NDArrays:
        return [v.cpu().numpy() for v in self.model.state_dict().values()]

    def set_parameters(self, parameters: NDArrays):
        state_dict = dict(zip(
            self.model.state_dict().keys(),
            [torch.tensor(p) for p in parameters]
        ))
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters: NDArrays, config: Dict) -> Tuple[NDArrays, int, Dict]:
        self.set_parameters(parameters)
        self.model.train()

        epochs = int(config.get('epochs', 3))
        lr     = float(config.get('lr', 1e-4))

        loader = DataLoader(
            ChestXrayDataset(self.dataframe, transform=train_transform),
            batch_size         = BATCH_SIZE,
            shuffle            = True,
            num_workers        = NUM_WORKERS,
            pin_memory         = True,
            persistent_workers = True,
            prefetch_factor    = PREFETCH
        )

        criterion = nn.BCEWithLogitsLoss()
        optimiser = torch.optim.Adam(self.model.parameters(), lr=lr)

        total_loss, total_samples = 0.0, 0
        for _ in range(epochs):
            for images, labels in loader:
                images  = images.to(self.device, non_blocking=True)
                labels  = labels.to(self.device, non_blocking=True).unsqueeze(1)
                outputs = self.model(images)
                loss    = criterion(outputs, labels)
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
                total_loss    += loss.item() * len(labels)
                total_samples += len(labels)

        avg_loss = total_loss / total_samples
        return (
            self.get_parameters(config={}),
            len(self.dataframe),
            {'loss': float(avg_loss), 'client_name': self.name}
        )

    def evaluate(self, parameters: NDArrays, config: Dict) -> Tuple[float, int, Dict]:
        self.set_parameters(parameters)
        m = evaluate_client(self.model, self.val_dataframe, self.device)
        m['client_name'] = self.name
        return float(1.0 - (m['auc'] if not np.isnan(m['auc']) else 0.5)), \
               len(self.val_dataframe), m

print("✓ HospitalClient defined")
print("  fit()      → trains on train_clients")
print("  evaluate() → validates on val_clients after each round")

✓ HospitalClient defined
  fit()      → trains on train_clients
  evaluate() → validates on val_clients after each round


# Federated Learning — Method 3: Ada-FFL (Cong et al. 2023)

**This notebook runs Ada-FFL only on the full dataset.**
Ada-FFL extends q-FedAvg by computing q adaptively each round using Frobenius
distance between local and global model weights — eliminates manual q tuning.
Citation: Cong et al. 2023 (CAAI Transactions on Intelligence Technology)

In [ ]:
# CELL — q-FedAvg client
# Overrides fit() to compute F_k(w_t) BEFORE local training
# Pre-loss loader uses num_workers=0 to avoid Ray worker deadlock

class QFedAvgHospitalClient(HospitalClient):

    def fit(self, parameters, config):
        self.set_parameters(parameters)

        epochs    = int(config.get('epochs', 3))
        lr        = float(config.get('lr', 1e-4))
        criterion = nn.BCEWithLogitsLoss()

        # Pre-loss loader — num_workers=0, no persistent_workers
        pre_loader = DataLoader(
            ChestXrayDataset(self.dataframe, transform=train_transform),
            batch_size  = BATCH_SIZE,
            shuffle     = False,
            num_workers = 0,
            pin_memory  = False,
        )

        # F_k(w_t) — pre-training loss [Li et al. 2020 Algorithm 2]
        self.model.eval()
        loss_sum, n = 0.0, 0
        with torch.no_grad():
            for images, labels in pre_loader:
                images = images.to(self.device, non_blocking=True)
                labels = labels.to(self.device, non_blocking=True).unsqueeze(1)
                loss_sum += criterion(self.model(images), labels).item() * len(labels)
                n        += len(labels)
        loss_before = loss_sum / n

        # Training loader — full settings
        train_loader = DataLoader(
            ChestXrayDataset(self.dataframe, transform=train_transform),
            batch_size         = BATCH_SIZE,
            shuffle            = True,
            num_workers        = NUM_WORKERS,
            pin_memory         = True,
            persistent_workers = True,
            prefetch_factor    = PREFETCH
        )

        self.model.train()
        optimiser = torch.optim.Adam(self.model.parameters(), lr=lr)
        total_loss, total_samples = 0.0, 0

        for _ in range(epochs):
            for images, labels in train_loader:
                images  = images.to(self.device, non_blocking=True)
                labels  = labels.to(self.device, non_blocking=True).unsqueeze(1)
                outputs = self.model(images)
                loss    = criterion(outputs, labels)
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
                total_loss    += loss.item() * len(labels)
                total_samples += len(labels)

        avg_loss = total_loss / total_samples
        return (
            self.get_parameters(config={}),
            len(self.dataframe),
            {'loss': float(avg_loss), 'client_name': self.name,
             'loss_before': float(loss_before)}
        )

print("✓ QFedAvgHospitalClient defined")
print("  Computes F_k(w_t) before local training [Li et al. 2020 Algorithm 2]")

✓ QFedAvgHospitalClient defined
  Computes F_k(w_t) before local training [Li et al. 2020 Algorithm 2]


In [ ]:
# CELL — q-FedAvg client factory

def make_qfedavg_client_fn(train_data_map, val_data_map, device):
    def client_fn(cid):
        name = HOSPITAL_NAMES[int(cid)]
        return QFedAvgHospitalClient(
            name          = name,
            dataframe     = train_data_map[name],
            val_dataframe = val_data_map[name],
            device        = device
        ).to_client()
    return client_fn

print("✓ Ada-IFFL client factory defined")

✓ Ada-IFFL client factory defined


In [ ]:
# CELL — Ada-FFL strategy — Cong et al. 2023
# Extends q-FedAvg: instead of fixed Q_PARAM, computes q_k adaptively
# each round using Frobenius distance between w_t and w_bar_k
# v_k = 1 - 2*||w_t - w_bar_k|| / (||w_t|| + ||w_bar_k||)
# q_k = beta * |1 - exp(-v_k)|
# Smaller model change → smaller q_k → prioritise utility
# Larger model change → larger q_k → prioritise fairness [Cong et al. 2023]

BETA = 1.0

class AdaIFFLStrategy(Strategy):

    def __init__(self, beta=1.0, L=1.0, num_clients=5):
        super().__init__()
        self.beta           = beta
        self.L              = L
        self.num_clients    = num_clients
        self._global_params = None

    def initialize_parameters(self, client_manager):
        ndarrays            = [v.cpu().numpy() for v in build_model().state_dict().values()]
        self._global_params = ndarrays
        return ndarrays_to_parameters(ndarrays)

    def configure_fit(self, server_round, parameters, client_manager):
        self._global_params = parameters_to_ndarrays(parameters)
        ins     = FitIns(parameters, {'epochs': 3, 'lr': 1e-4})
        sampled = client_manager.sample(num_clients=self.num_clients,
                                        min_num_clients=self.num_clients)
        return [(c, ins) for c in sampled]

    def _compute_q(self, w_t, w_bar_k):
        # Frobenius distance — Cong et al. 2023 Eq.3
        norm_diff = sum(np.linalg.norm(w - wb) for w, wb in zip(w_t, w_bar_k))
        norm_wt   = sum(np.linalg.norm(w)  for w  in w_t)
        norm_wb   = sum(np.linalg.norm(wb) for wb in w_bar_k)
        denom     = norm_wt + norm_wb
        v_k       = 1.0 - (2.0 * norm_diff / denom) if denom > 0 else 0.0
        v_k       = float(np.clip(v_k, 0.0, 1.0))
        q_k       = self.beta * abs(1.0 - np.exp(-v_k))
        return max(q_k, 1e-10)

    def aggregate_fit(self, server_round, results, failures):
        if not results:
            return None, {}

        w_t       = self._global_params
        sum_Delta = None
        sum_h     = 0.0

        for _, fit_res in results:
            w_bar_k = parameters_to_ndarrays(fit_res.parameters)
            loss_k  = max(float(fit_res.metrics.get('loss_before', 1.0)), 1e-10)

            q_k = self._compute_q(w_t, w_bar_k)

            delta_w_k = [self.L * (w - wb) for w, wb in zip(w_t, w_bar_k)]
            Delta_k   = [(loss_k ** q_k) * dw for dw in delta_w_k]
            norm_sq   = float(sum(np.sum(dw ** 2) for dw in delta_w_k))
            h_k       = (q_k * (loss_k ** (q_k - 1)) * norm_sq
                         + self.L * (loss_k ** q_k))

            sum_Delta = Delta_k if sum_Delta is None else \
                        [sd + dk for sd, dk in zip(sum_Delta, Delta_k)]
            sum_h    += h_k

        if sum_h == 0.0 or sum_Delta is None:
            return ndarrays_to_parameters(w_t), {}

        w_new               = [w - sd / sum_h for w, sd in zip(w_t, sum_Delta)]
        self._global_params = w_new
        return ndarrays_to_parameters(w_new), {}

    def aggregate_evaluate(self, server_round, results, failures):
        if not results:
            return None, {}

        print(f"\n── Round {server_round}/{ROUNDS} Validation ──")
        for _, res in results:
            name = res.metrics.get('client_name', '?')
            auc  = res.metrics.get('auc', float('nan'))
            fnr  = res.metrics.get('fnr', float('nan'))
            print(f"  {name:<14} AUC: {auc:.4f}  FNR: {fnr:.4f}")
        aucs       = [r.metrics.get('auc', float('nan')) for _, r in results]
        valid_aucs = [a for a in aucs if not (a != a)]
        if valid_aucs:
            print(f"  Mean AUC : {sum(valid_aucs)/len(valid_aucs):.4f} | "
                  f"Variance : {float(np.var(valid_aucs)):.6f}")

        total_loss = sum(r.loss * r.num_examples for _, r in results)
        total_n    = sum(r.num_examples for _, r in results)
        return total_loss / total_n, {}

    def configure_evaluate(self, server_round, parameters, client_manager):
        ins     = EvaluateIns(parameters, {})
        sampled = client_manager.sample(num_clients=self.num_clients,
                                        min_num_clients=self.num_clients)
        return [(c, ins) for c in sampled]

    def evaluate(self, server_round, parameters):
        return None

print(f"✓ AdaIFFLStrategy defined [Cong et al. 2023 Algorithm 2]")
print(f"  beta={BETA}  L=1.0")
print(f"  Individual adaptive q per client per round via Frobenius distance")
print(f"  Per-round validation output enabled")

✓ AdaIFFLStrategy defined [Cong et al. 2023 Algorithm 2]
  beta=1.0  L=1.0
  Individual adaptive q per client per round via Frobenius distance
  Per-round validation output enabled


In [ ]:
# CELL — Run Ada-FFL simulation

import os, logging
os.environ['RAY_SILENT_MODE'] = '1'
logging.getLogger('flwr').setLevel(logging.ERROR)

print("=" * 50)
print("Ada-IFFL — Cong et al. 2023 Algorithm 2")
print(f"Rounds: {ROUNDS} | Epochs/round: 3 | Clients: {NUM_CLIENTS}")
print(f"beta={BETA}  L=1.0 | Split: 85/10/5 | Batch: {BATCH_SIZE}")
print("=" * 50)

t0              = time.time()
adaiffl_strategy = AdaIFFLStrategy(beta=BETA, L=1.0, num_clients=NUM_CLIENTS)

adaiffl_history = fl.simulation.start_simulation(
    client_fn        = make_qfedavg_client_fn(train_clients, val_clients, device),
    num_clients      = NUM_CLIENTS,
    config           = fl.server.ServerConfig(num_rounds=ROUNDS),
     strategy         = adaiffl_strategy,
    client_resources = {'num_gpus': 1.0}
)

print(f"\n{'='*50}")
print(f"✓ Ada-FFL complete in {(time.time()-t0)/60:.1f} minutes")
print(f"\nLoss per round:")
for rnd, loss in adaiffl_history.losses_distributed:
    print(f"  Round {rnd:>2} : {loss:.4f}")

Ada-IFFL — Cong et al. 2023 Algorithm 2
Rounds: 10 | Epochs/round: 3 | Clients: 5
beta=1.0  L=1.0 | Split: 85/10/5 | Batch: 512


2026-06-30 13:33:35,959	INFO worker.py:2012 -- Started a local Ray instance.
(ClientAppActor pid=11136) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=11136) 
(ClientAppActor pid=11136)             This is a deprecated feature. It will be removed
(ClientAppActor pid=11136)             entirely in future versions of Flower.
(ClientAppActor pid=11136)         
(ClientAppActor pid=11136) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=11136) is multi-threaded, use of fork() may lead to deadlocks in the child.
(ClientAppActor pid=11136)   self.pid = os.fork()
(ClientAppActor pid=11136) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Pa


── Round 1/10 Validation ──
  Hospital_D     AUC: 0.5483  FNR: 0.9528
  Hospital_B     AUC: 0.2706  FNR: 0.9492
  Hospital_E     AUC: 0.5213  FNR: 0.9600
  Hospital_C     AUC: 0.5689  FNR: 0.9556
  Hospital_A     AUC: 0.5771  FNR: 0.9564
  Mean AUC : 0.4972 | Variance : 0.013213


(ClientAppActor pid=11136) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=11136) 
(ClientAppActor pid=11136)             This is a deprecated feature. It will be removed
(ClientAppActor pid=11136)             entirely in future versions of Flower.
(ClientAppActor pid=11136)         
(ClientAppActor pid=11136) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=11136) is multi-threaded, use of fork() may lead to deadlocks in the child.
(ClientAppActor pid=11136)   self.pid = os.fork()
(ClientAppActor pid=11136) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app impor


── Round 2/10 Validation ──
  Hospital_D     AUC: 0.5487  FNR: 0.9528
  Hospital_A     AUC: 0.5773  FNR: 0.9553
  Hospital_B     AUC: 0.2709  FNR: 0.9482
  Hospital_E     AUC: 0.5217  FNR: 0.9600
  Hospital_C     AUC: 0.5692  FNR: 0.9556
  Mean AUC : 0.4976 | Variance : 0.013213


...


── Round 8/10 Validation ──
  Hospital_E     AUC: 0.5231  FNR: 0.9600
  Hospital_B     AUC: 0.2750  FNR: 0.9444
  Hospital_A     AUC: 0.5787  FNR: 0.9536
  Hospital_D     AUC: 0.5505  FNR: 0.9528
  Hospital_C     AUC: 0.5708  FNR: 0.9522
  Mean AUC : 0.4996 | Variance : 0.012983


(ClientAppActor pid=11136) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=11136) 
(ClientAppActor pid=11136)             This is a deprecated feature. It will be removed
(ClientAppActor pid=11136)             entirely in future versions of Flower.
(ClientAppActor pid=11136)         
(ClientAppActor pid=11136) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=11136) is multi-threaded, use of fork() may lead to deadlocks in the child.
(ClientAppActor pid=11136)   self.pid = os.fork()
(ClientAppActor pid=11136) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app impor


── Round 9/10 Validation ──
  Hospital_A     AUC: 0.5789  FNR: 0.9531
  Hospital_E     AUC: 0.5234  FNR: 0.9600
  Hospital_D     AUC: 0.5509  FNR: 0.9528
  Hospital_C     AUC: 0.5709  FNR: 0.9488
  Hospital_B     AUC: 0.2753  FNR: 0.9435
  Mean AUC : 0.4999 | Variance : 0.012976


(ClientAppActor pid=11136) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=11136) 
(ClientAppActor pid=11136)             This is a deprecated feature. It will be removed
(ClientAppActor pid=11136)             entirely in future versions of Flower.
(ClientAppActor pid=11136)         
(ClientAppActor pid=11136) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=11136) is multi-threaded, use of fork() may lead to deadlocks in the child.
(ClientAppActor pid=11136)   self.pid = os.fork()
(ClientAppActor pid=11136) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app impor


── Round 10/10 Validation ──
  Hospital_C     AUC: 0.5711  FNR: 0.9488
  Hospital_A     AUC: 0.5791  FNR: 0.9531
  Hospital_D     AUC: 0.5511  FNR: 0.9528
  Hospital_E     AUC: 0.5237  FNR: 0.9600
  Hospital_B     AUC: 0.2765  FNR: 0.9435
  Mean AUC : 0.5003 | Variance : 0.012887

✓ Ada-FFL complete in 671.2 minutes

Loss per round:
  Round  1 : 0.4567
  Round  2 : 0.4565
  Round  3 : 0.4561
  Round  4 : 0.4559
  Round  5 : 0.4556
  Round  6 : 0.4553
  Round  7 : 0.4550
  Round  8 : 0.4547
  Round  9 : 0.4546
  Round 10 : 0.4543


In [ ]:
for rnd, loss in adaiffl_history.losses_distributed:
    print(f"  Round {rnd:>2} : {loss:.4f}")

  Round  1 : 0.4567
  Round  2 : 0.4565
  Round  3 : 0.4561
  Round  4 : 0.4559
  Round  5 : 0.4556
  Round  6 : 0.4553
  Round  7 : 0.4550
  Round  8 : 0.4547
  Round  9 : 0.4546
  Round 10 : 0.4543


In [ ]:
# CELL — Evaluate Ada-IFFL per client

import gc, ray
if ray.is_initialized():
    ray.shutdown()
torch.cuda.empty_cache()
gc.collect()

adaiffl_metrics      = {}
adaiffl_final_params = adaiffl_strategy._global_params

for name, data in test_clients.items():
    model = build_model().to(device)
    model.load_state_dict(dict(zip(
        model.state_dict().keys(),
        [torch.tensor(p) for p in adaiffl_final_params]
    )))
    adaiffl_metrics[name] = evaluate_client(model, data, device)
    del model
    torch.cuda.empty_cache()

rows = []
for name in HOSPITAL_NAMES:
    m = adaiffl_metrics[name]
    rows.append({
        'Client'   : name,
        'AUC'      : m['auc'],
        'FNR'      : m['fnr'],
        'Accuracy' : m['accuracy'],
        'AUC_M'    : m.get('auc_M', 'N/A'),
        'FNR_M'    : m.get('fnr_M', 'N/A'),
        'AUC_F'    : m.get('auc_F', 'N/A'),
        'FNR_F'    : m.get('fnr_F', 'N/A'),
    })

df_adaiffl = pd.DataFrame(rows).set_index('Client')
print("=== Ada-IFFL Results — Cong et al. 2023 Algorithm 2 ===\n")
print(df_adaiffl.to_string())

valid_aucs = [adaiffl_metrics[n]['auc'] for n in HOSPITAL_NAMES
              if not (adaiffl_metrics[n]['auc'] != adaiffl_metrics[n]['auc'])]
auc_var = np.var(valid_aucs) if valid_aucs else float('nan')
print(f"\nAUC Variance (valid hospitals only) : {auc_var:.6f}")
print(f"Valid hospitals : {len(valid_aucs)}/5")

=== Ada-IFFL Results — Cong et al. 2023 Algorithm 2 ===

               AUC     FNR  Accuracy   AUC_M   FNR_M   AUC_F   FNR_F
Client                                                              
Hospital_A  0.5633  0.9575     42.07  0.5570  0.9575  0.5721  0.9574
Hospital_B  0.8049  0.9343      6.74     NaN  0.9379  0.8152  0.9289
Hospital_C  0.5992  0.9538     91.48  0.5958  0.9737  0.6023  0.9259
Hospital_D  0.5312  0.9464     30.26  0.5525  0.9412  0.4773  0.9545
Hospital_E  0.5265  0.9286     87.50  0.4552  0.8889  0.6506  1.0000

AUC Variance (valid hospitals only) : 0.010668
Valid hospitals : 5/5


In [ ]:
# CELL — Save Ada-IFFL results to Drive

SAVE_DIR = '/content/drive/MyDrive/dissertation/results'
os.makedirs(SAVE_DIR, exist_ok=True)

with open(f'{SAVE_DIR}/adaiffl_full_metrics.json', 'w') as f:
    json.dump(adaiffl_metrics, f, indent=2, default=str)

torch.save(
    dict(zip(build_model().state_dict().keys(),
             [torch.tensor(v) for v in adaiffl_final_params])),
    f'{SAVE_DIR}/adaiffl_full_model.pth'
)

print("✓ Ada-IFFL metrics saved → adaiffl_full_metrics.json")
print("✓ Ada-IFFL model saved  → adaiffl_full_model.pth")

✓ Ada-IFFL metrics saved → adaiffl_full_metrics.json
✓ Ada-IFFL model saved  → adaiffl_full_model.pth
